Laboratorio 5: Manejo de errores
Proyecto: Análisis del Rendimiento Académico

Integrante:
Rumay Avedaño, Meylin

1. Definir el esquema

In [0]:
from pyspark.sql.types import StructType, StringType, IntegerType, FloatType

estudiantes_schema = (
    StructType()
    .add("Student_ID", IntegerType())
    .add("Semester_ID", IntegerType())
    .add("Age", IntegerType())
    .add("Gender", StringType())
    .add("Region_Type", StringType())
    .add("Family_Size", IntegerType())
    .add("Major_Subject", StringType())
    .add("University_Name", StringType())
    .add("Home_City", StringType())
    .add("Parent_Education", StringType())
    .add("Family_Income_Level", StringType())
    .add("Internet_Quality", FloatType())
    .add("Study_Space_Quality", FloatType())
    .add("Previous_GPA", FloatType())
    .add("Number_of_Failed_Courses", IntegerType())
    .add("Total_Credits_Earned", IntegerType())
    .add("Weekly_Study_Hours", FloatType())
    .add("Attendance_Rate", FloatType())
    .add("Library_Visits_Per_Month", IntegerType())
    .add("Extracurricular_Hours", FloatType())
    .add("Sleep_Hours", FloatType())
    .add("Social_Media_Usage_Hours", FloatType())
    .add("Stress_Level", FloatType())
    .add("Motivation_Score", FloatType())
    .add("Self_Efficacy_Score", FloatType())
    .add("Midterm_Mark", FloatType())
    .add("Final_Exam_Score", FloatType())
    .add("_rescued_data", StringType())
)

print("Esquema definido correctamente")

2. Leer los JSON con detección de errores

In [0]:
from pyspark.sql.functions import col

df_estudiantes = (
    spark.read
    .format("json")
    .option("multiLine", "true")
    .option("columnNameOfCorruptRecord", "_rescued_data")
    .option("badRecordsPath", "/Volumes/proyecto_estudiantes1/bronze/volumen1/input/json/bad_records/")
    .load("/Volumes/proyecto_estudiantes1/bronze/volumen1/input/json/")
    .withColumn("archivo_origen", col("_metadata.file_path"))
)

print("Datos cargados correctamente")
print(f"Total registros: {df_estudiantes.count()}")
df_estudiantes.printSchema()

3. Ver los datos cargados

In [0]:
display(df_estudiantes)

4. Separar datos válidos e inválidos

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StringType

try:
    # Primero eliminar tablas anteriores si existen
    spark.sql("DROP TABLE IF EXISTS proyecto_estudiantes1.silver.estudiantes_validos")
    spark.sql("DROP TABLE IF EXISTS proyecto_estudiantes1.silver.estudiantes_invalidos")

    # Agregar columna _rescued_data si no existe
    if "_rescued_data" not in df_estudiantes.columns:
        df_estudiantes2 = df_estudiantes.withColumn("_rescued_data", lit(None).cast(StringType()))
    else:
        df_estudiantes2 = df_estudiantes

    # Separar válidos e inválidos
    df_validos = df_estudiantes2.filter(
        col("Student_ID").isNotNull() & col("Age").between(16, 40)
    )
    df_invalidos = df_estudiantes2.filter(
        col("Student_ID").isNull() | ~col("Age").between(16, 40)
    )

    # Guardar como nuevas tablas Delta
    df_validos.write.format("delta").mode("overwrite").saveAsTable(
        "proyecto_estudiantes1.silver.estudiantes_validos"
    )
    df_invalidos.write.format("delta").mode("overwrite").saveAsTable(
        "proyecto_estudiantes1.silver.estudiantes_invalidos"
    )

    print("Escritura realizada con éxito.")
    print(f"Válidos: {df_validos.count()} | Inválidos: {df_invalidos.count()}")

except Exception as e:
    print(f"Error durante la escritura: {str(e)}")

5. Validar cuántos válidos e inválidos hay

In [0]:
%sql
SELECT 'Validos' AS tipo, COUNT(*) AS cantidad FROM proyecto_estudiantes1.silver.estudiantes_validos
UNION ALL
SELECT 'Invalidos' AS tipo, COUNT(*) AS cantidad FROM proyecto_estudiantes1.silver.estudiantes_invalidos;

6. Ver los registros inválidos

In [0]:
%sql
SELECT * FROM proyecto_estudiantes1.silver.estudiantes_invalidos;